<a href="https://colab.research.google.com/github/mariea-aashif/Statistical-Learning-e22374/blob/main/assignment_7b__gaussian_mixture_model_clustering_as_conditional_updating.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bayesian Estimation of a User Ability Parameter from Item Responses

This assignment uses the two-parameter logistic (2PL) item response theory model:

$$
P(Y_i = 1 \mid \Theta = \theta)
=
p_i(\theta)
=
\frac{1}{1 + e^{-a_i(\theta-b_i)}},
$$

where $a_i>0$ is the item discrimination parameter, $b_i$ is the item difficulty parameter, and $\theta$ is the user's latent ability.

## Task 1: Visualizing the Mechanics

We visualize the probability of a correct answer,

$$
P(Y_i=1\mid\Theta=\theta)
=
\frac{1}{1+e^{-a_i(\theta-b_i)}},
$$

against the ability parameter $\theta$.

I use $a_i=1.0$ with three difficulty values, $b_i\in\{-1,0,1\}$, to examine the effect of difficulty. I also include an item with $a_i=2.0$ and $b_i=0$ to compare the effect of a higher discrimination parameter.

In [17]:
import numpy as np
import plotly.graph_objects as go

# Grid of possible latent ability values
theta = np.linspace(-4, 4, 500)

def probability_correct(theta, a, b):
    """2PL probability of a correct response."""
    return 1 / (1 + np.exp(-a * (theta - b)))

fig = go.Figure()

# Fix a = 1 and vary b to show the effect of difficulty
a = 1.0
for b in [-1.0, 0.0, 1.0]:
    fig.add_trace(
        go.Scatter(
            x=theta,
            y=probability_correct(theta, a, b),
            mode="lines",
            name=f"a = {a}, b = {b}"
        )
    )

# Compare with a more discriminating item
fig.add_trace(
    go.Scatter(
        x=theta,
        y=probability_correct(theta, a=2.0, b=0.0),
        mode="lines",
        line=dict(dash="dash", width=3),
        name="a = 2.0, b = 0.0"
    )
)

fig.add_hline(
    y=0.5,
    line_dash="dot",
    line_color="gray",
    annotation_text="P(Yᵢ = 1) = 0.5"
)

fig.update_layout(
    title="2PL Item Response Curves",
    xaxis_title="Latent ability, θ",
    yaxis_title="Probability of a correct response",
    yaxis=dict(range=[0, 1]),
    template="plotly_white",
    width=900,
    height=550,
    legend_title="Item parameters"
)

fig.show()

### Interpretation

For every item, the response probability equals $0.5$ when ability equals the item's difficulty:

$$
p_i(b_i)=0.5.
$$

Therefore, $b_i$ controls the horizontal position of the curve. As $b_i$ increases, the curve shifts to the right. This means that a higher ability level $\theta$ is required to obtain the same probability of a correct answer. Hence, larger values of $b_i$ represent more difficult items.

For example, the curve with $b_i=1$ is farther to the right than the curve with $b_i=-1$, so it represents a harder item.

The discrimination parameter $a_i$ controls the steepness of the curve. The curve with $a_i=2$ changes more sharply around $b_i=0$ than the curve with $a_i=1$. Thus, a highly discriminating item better separates users whose abilities are just below and just above the item's difficulty level.

## Task 2: Sequential Likelihood Contribution

For the $k$-th item, the observed response is $y_k\in\{0,1\}$. Conditional on ability $\theta$, this response follows a Bernoulli distribution with success probability

$$
p_k(\theta)
=
P(Y_k=1\mid\Theta=\theta)
=
\frac{1}{1+e^{-a_k(\theta-b_k)}}.
$$

Therefore, the likelihood contribution of one new response $y_k$ is

$$
L(y_k\mid\theta)
=
P(Y_k=y_k\mid\Theta=\theta)
=
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k}.
$$

Substituting the 2PL response probability gives

$$
L(y_k\mid\theta)
=
\left[
\frac{1}{1+e^{-a_k(\theta-b_k)}}
\right]^{y_k}
\left[
1-
\frac{1}{1+e^{-a_k(\theta-b_k)}}
\right]^{1-y_k}.
$$

This formula covers both possible observed outcomes:

$$
L(y_k\mid\theta)
=
\begin{cases}
p_k(\theta), & \text{if } y_k=1,\\
1-p_k(\theta), & \text{if } y_k=0.
\end{cases}
$$

Assuming that responses are conditionally independent given $\Theta=\theta$, the joint likelihood for the running response history

$$
\mathbf{y}^{(k)}=(y_1,y_2,\ldots,y_k)
$$

is

$$
L(\mathbf{y}^{(k)}\mid\theta)
=
\prod_{j=1}^{k}
[p_j(\theta)]^{y_j}
[1-p_j(\theta)]^{1-y_j},
$$

where

$$
p_j(\theta)
=
\frac{1}{1+e^{-a_j(\theta-b_j)}}.
$$

Thus, each newly observed item response adds one Bernoulli likelihood factor to the running likelihood.

## Task 3: Mathematical Formulation of the Running Update

Let

$$
f_{\Theta\mid\mathbf{Y}^{(k-1)}}
\left(
\theta\mid\mathbf{y}^{(k-1)}
\right)
$$

denote the posterior density after observing the first $k-1$ responses. Before observing response $y_k$, this posterior becomes the prior distribution for step $k$.

By Bayes' theorem, the updated posterior after observing $y_k$ is proportional to the previous posterior multiplied by the likelihood contribution of the new response:

$$
f_{\Theta\mid\mathbf{Y}^{(k)}}
\left(
\theta\mid\mathbf{y}^{(k)}
\right)
\propto
L(y_k\mid\theta)
\,
f_{\Theta\mid\mathbf{Y}^{(k-1)}}
\left(
\theta\mid\mathbf{y}^{(k-1)}
\right).
$$

Using the Bernoulli likelihood under the 2PL model, this becomes

$$
f_{\Theta\mid\mathbf{Y}^{(k)}}
\left(
\theta\mid\mathbf{y}^{(k)}
\right)
\propto
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k}
f_{\Theta\mid\mathbf{Y}^{(k-1)}}
\left(
\theta\mid\mathbf{y}^{(k-1)}
\right),
$$

where

$$
p_k(\theta)
=
\frac{1}{1+e^{-a_k(\theta-b_k)}}.
$$

The fully normalized posterior is therefore

$$
f_{\Theta\mid\mathbf{Y}^{(k)}}
\left(
\theta\mid\mathbf{y}^{(k)}
\right)
=
\frac{
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k}
f_{\Theta\mid\mathbf{Y}^{(k-1)}}
\left(
\theta\mid\mathbf{y}^{(k-1)}
\right)
}{
\int_{-\infty}^{\infty}
[p_k(t)]^{y_k}
[1-p_k(t)]^{1-y_k}
f_{\Theta\mid\mathbf{Y}^{(k-1)}}
\left(
t\mid\mathbf{y}^{(k-1)}
\right)
\,dt
}.
$$

The denominator is the normalizing constant. It ensures that the updated posterior density integrates to one:

$$
\int_{-\infty}^{\infty}
f_{\Theta\mid\mathbf{Y}^{(k)}}
\left(
\theta\mid\mathbf{y}^{(k)}
\right)
\,d\theta
=
1.
$$

Initially, before any responses are observed, the prior is

$$
f_{\Theta}^{(0)}(\theta)
=
\frac{1}{\sqrt{2\pi}}
\exp\left(-\frac{\theta^2}{2}\right),
$$

which corresponds to

$$
\Theta\sim\mathscr{N}(0,1).
$$

## Task 4: Dynamic Shifting

Suppose the user answers item $k$ correctly, so that $y_k=1$. The likelihood contribution is then

$$
L(y_k=1\mid\theta)=p_k(\theta)
=
\frac{1}{1+e^{-a_k(\theta-b_k)}}.
$$

The sequential posterior update becomes

$$
f_{\Theta\mid\mathbf{Y}^{(k)}}
(\theta\mid\mathbf{y}^{(k)})
\propto
p_k(\theta)
f_{\Theta\mid\mathbf{Y}^{(k-1)}}
(\theta\mid\mathbf{y}^{(k-1)}).
$$

For a difficult item, $b_k$ is large. The probability $p_k(\theta)$ is then relatively small for low and moderate values of $\theta$, but it becomes larger for high values of $\theta$. Therefore, multiplying the previous posterior by $p_k(\theta)$ downweights low-ability values and gives relatively more weight to high-ability values.

Mathematically, for a correct answer,

$$
\frac{d}{d\theta}\log p_k(\theta)
=
a_k[1-p_k(\theta)] > 0.
$$

Thus, the log-likelihood from a correct answer has a positive slope with respect to $\theta$. When it is added to the log of the previous posterior, it pushes the posterior mode, or peak, towards larger values of $\theta$.

Hence, correctly answering a highly difficult item shifts the running posterior density to the right. This indicates an increased belief that the user has a high latent ability.

## Task 5: Tracking Certainty and Sharpness

The discrimination parameter $a_k$ controls how strongly the current item response distinguishes between users with different ability levels.

The response probability is

$$
p_k(\theta)
=
\frac{1}{1+e^{-a_k(\theta-b_k)}}.
$$

The information provided by an item is related to

$$
I_k(\theta)
=
a_k^2 p_k(\theta)[1-p_k(\theta)].
$$

This quantity is largest when $\theta$ is close to the item difficulty $b_k$, because then

$$
p_k(\theta)\approx 0.5.
$$

When $a_k$ is very large, the item response curve becomes steep near $b_k$. A correct or incorrect response then gives strong evidence about whether the user's ability is above or below the item difficulty. Consequently, the posterior is usually updated more sharply and its variance tends to decrease more.

When $a_k$ is very small, the item response curve is nearly flat. The probability of a correct answer changes only slightly as $\theta$ changes, so the response contains little information about ability. The posterior changes only slightly, and its variance remains relatively large.

Therefore, high-discrimination items generally produce sharper posterior distributions, while low-discrimination items produce weaker updates. The strongest reduction in uncertainty occurs when the item difficulty is close to the current plausible ability range.

## Task 6: Numerical Implementation of a Running Grid

The posterior distribution does not have a simple closed-form expression because the logistic likelihood is not conjugate to the normal prior. Therefore, the posterior can be approximated numerically on a fixed grid of possible ability values.

First, choose a grid:

$$
\theta_1,\theta_2,\ldots,\theta_m,
$$

for example over the interval $[-4,4]$. This range covers almost all of the mass of the standard normal prior.

At step $0$, evaluate the standard normal prior density at every grid point:

$$
f^{(0)}(\theta_g)
=
\frac{1}{\sqrt{2\pi}}
\exp\left(-\frac{\theta_g^2}{2}\right).
$$

After observing response $y_k$, calculate the likelihood at every grid point:

$$
L(y_k\mid\theta_g)
=
[p_k(\theta_g)]^{y_k}
[1-p_k(\theta_g)]^{1-y_k}.
$$

Then compute the unnormalized posterior:

$$
\widetilde f^{(k)}(\theta_g)
=
L(y_k\mid\theta_g)
f^{(k-1)}(\theta_g).
$$

To normalize numerically, approximate the integral using the trapezoidal rule:

$$
Z_k
\approx
\int_{-4}^{4}
\widetilde f^{(k)}(\theta)\,d\theta.
$$

The normalized posterior is

$$
f^{(k)}(\theta_g)
=
\frac{\widetilde f^{(k)}(\theta_g)}{Z_k}.
$$

Computationally, this normalization is performed using

$$
Z_k \approx \texttt{np.trapezoid(unnormalized\_posterior, theta\_grid)}.
$$

This update is repeated sequentially after every new item response. The normalized posterior at step $k$ becomes the prior for step $k+1$.

In [18]:
import numpy as np

# Fixed grid of plausible ability values
theta_grid = np.linspace(-4, 4, 2001)

def normal_prior(theta):
    """Standard normal prior density."""
    return (1 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * theta**2)

def item_probability(theta, a, b):
    """2PL probability of a correct response."""
    return 1 / (1 + np.exp(-a * (theta - b)))

def update_posterior(theta_grid, previous_posterior, y, a, b):
    """
    Update a grid-based posterior after one observed response.
    y = 1 for a correct response and y = 0 for an incorrect response.
    """
    p = item_probability(theta_grid, a, b)
    likelihood = (p ** y) * ((1 - p) ** (1 - y))

    unnormalized_posterior = likelihood * previous_posterior
    normalizing_constant = np.trapezoid(
        unnormalized_posterior,
        theta_grid
    )

    posterior = unnormalized_posterior / normalizing_constant
    return posterior

# Initial posterior equals the prior, normalized over the chosen grid
posterior = normal_prior(theta_grid)
posterior = posterior / np.trapezoid(posterior, theta_grid)

## Task 7: Evaluating Convergence over the Timeline

In this simulation, the true but unknown user ability is set to

$$
\theta_{\text{true}}=0.75.
$$

A sequence of $n=20$ items is generated. For each item,

$$
b_k\sim\mathscr{N}(0,1)
$$

and

$$
a_k\sim\text{Uniform}(0.5,2.0).
$$

The response is simulated from the 2PL model. In particular,

$$
Y_k=
\begin{cases}
1, & \text{if } U_k < p_k(\theta_{\text{true}}),\\
0, & \text{otherwise},
\end{cases}
$$

where

$$
U_k\sim\text{Uniform}(0,1).
$$

At every step, two running estimators are calculated:

$$
\widehat{\theta}_{\mathrm{Bayes}}^{(k)}
=
\int_{-\infty}^{\infty}
\theta f^{(k)}(\theta)\,d\theta,
$$

and

$$
\widehat{\theta}_{\mathrm{MAP}}^{(k)}
=
\underset{\theta}{\operatorname{arg\,max}}
\ f^{(k)}(\theta).
$$

The posterior mean is approximated using the trapezoidal rule, while the MAP estimate is approximated by the grid value with the largest posterior density.

In [19]:
import numpy as np
import plotly.graph_objects as go

# Reproducible random simulation
rng = np.random.default_rng(seed=42)

# Simulation settings
theta_true = 0.75
n_items = 20

# Fixed ability grid
theta_grid = np.linspace(-4, 4, 2001)

def normal_prior(theta):
    """Standard normal prior density."""
    return (1 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * theta**2)

def item_probability(theta, a, b):
    """2PL probability of a correct response."""
    return 1 / (1 + np.exp(-a * (theta - b)))

def update_posterior(theta_grid, previous_posterior, y, a, b):
    """Perform one sequential grid-based Bayesian update."""
    p = item_probability(theta_grid, a, b)
    likelihood = (p ** y) * ((1 - p) ** (1 - y))

    unnormalized_posterior = likelihood * previous_posterior
    normalizing_constant = np.trapezoid(
        unnormalized_posterior,
        theta_grid
    )

    return unnormalized_posterior / normalizing_constant

# Initialize with the standard normal prior
posterior = normal_prior(theta_grid)
posterior = posterior / np.trapezoid(posterior, theta_grid)

# Store results beginning at step 0
steps = [0]
posterior_means = [
    np.trapezoid(theta_grid * posterior, theta_grid)
]
map_estimates = [
    theta_grid[np.argmax(posterior)]
]

# Optional: store the simulated item parameters and responses
item_difficulties = []
item_discriminations = []
responses = []

# Sequential simulation and updating
for k in range(1, n_items + 1):
    # Generate random item parameters
    b_k = rng.normal(loc=0, scale=1)
    a_k = rng.uniform(0.5, 2.0)

    # Simulate the user's response using the true ability
    p_true = item_probability(theta_true, a_k, b_k)
    y_k = int(rng.uniform(0, 1) < p_true)

    # Update the posterior distribution
    posterior = update_posterior(
        theta_grid,
        posterior,
        y_k,
        a_k,
        b_k
    )

    # Calculate the posterior mean and MAP estimate
    posterior_mean = np.trapezoid(theta_grid * posterior, theta_grid)
    map_estimate = theta_grid[np.argmax(posterior)]

    # Store values
    steps.append(k)
    posterior_means.append(posterior_mean)
    map_estimates.append(map_estimate)

    item_difficulties.append(b_k)
    item_discriminations.append(a_k)
    responses.append(y_k)

# Display the simulated responses
print("Simulated responses:", responses)
print("Number of correct answers:", sum(responses), "out of", n_items)

# Plot convergence of the two running estimators
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode="lines+markers",
        name="Posterior Mean",
        line=dict(width=3)
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP Estimate",
        line=dict(width=3, dash="dash")
    )
)

fig.add_hline(
    y=theta_true,
    line_dash="dot",
    line_color="red",
    annotation_text="True ability = 0.75"
)

fig.update_layout(
    title="Convergence of Sequential Ability Estimates",
    xaxis_title="Item step, k",
    yaxis_title="Estimated ability",
    template="plotly_white",
    width=900,
    height=550,
    legend_title="Estimator"
)

fig.show()

Simulated responses: [0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1]
Number of correct answers: 13 out of 20


### Analysis

At step $k=0$, both estimators are close to $0$ because the initial prior is

$$
\Theta\sim\mathscr{N}(0,1).
$$

As item responses are observed, the posterior mean and MAP estimate update using the evidence supplied by the responses and the item parameters. In general, both estimates should move toward the true ability value,

$$
\theta_{\text{true}}=0.75.
$$

However, the movement is not necessarily monotonic. Random responses, especially unexpected correct or incorrect answers, may temporarily move an estimate away from the true ability. The item difficulty and discrimination parameters also determine how informative each response is.

As $k$ increases, the posterior typically becomes more concentrated because evidence accumulates over multiple responses. A narrower posterior distribution indicates greater confidence in the estimated user ability. Therefore, if the posterior mean and MAP estimate become more stable and remain close to $0.75$, the platform has increasing confidence in its measurement of the user's ability.

# Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

Let $\Theta=\theta$ denote the unknown click-through rate (CTR), where

$$
0\leq\theta\leq1.
$$

For each advertisement impression at step $k$, define

$$
Y_k=
\begin{cases}
1, & \text{if the user clicks the advertisement},\\
0, & \text{if the user does not click the advertisement}.
\end{cases}
$$

Conditional on the CTR $\Theta=\theta$, each interaction is modelled as a Bernoulli trial:

$$
P(Y_k=1\mid\Theta=\theta)=\theta.
$$

Before observing data, the prior distribution is

$$
\Theta\sim\text{Beta}(\alpha_0,\beta_0),
$$

with density

$$
f_{\Theta}^{(0)}(\theta)
=
\frac{1}{\mathrm{B}(\alpha_0,\beta_0)}
\theta^{\alpha_0-1}(1-\theta)^{\beta_0-1}.
$$

## Task 1: Structural Probability and Properties

The Beta distribution is defined on the interval $[0,1]$, so it is suitable for modelling an unknown probability such as a click-through rate.

The following three prior states are plotted:

- Uniform prior: $\text{Beta}(1,1)$
- Right-skewed prior: $\text{Beta}(2,8)$
- Left-skewed prior: $\text{Beta}(8,2)$

In [20]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Avoid the exact endpoints for numerical stability
theta = np.linspace(0.001, 0.999, 1000)

parameter_sets = [
    (1, 1, "Beta(1, 1): Uniform"),
    (2, 8, "Beta(2, 8): Right-skewed"),
    (8, 2, "Beta(8, 2): Left-skewed")
]

fig = go.Figure()

for alpha, beta_param, label in parameter_sets:
    density = beta.pdf(theta, alpha, beta_param)

    fig.add_trace(
        go.Scatter(
            x=theta,
            y=density,
            mode="lines",
            name=label,
            line=dict(width=3)
        )
    )

fig.update_layout(
    title="Beta Distribution Densities for Different Parameter Values",
    xaxis_title="Click-through rate, θ",
    yaxis_title="Probability density",
    template="plotly_white",
    width=900,
    height=550,
    legend_title="Distribution"
)

fig.show()

### Interpretation

For a Beta distribution,

$$
\mathbb{E}[\Theta]
=
\frac{\alpha}{\alpha+\beta}.
$$

For $\text{Beta}(1,1)$,

$$
\mathbb{E}[\Theta]=\frac{1}{2},
$$

and the density is constant across $[0,1]$. This represents no preference for low or high CTR values.

For $\text{Beta}(2,8)$,

$$
\mathbb{E}[\Theta]=\frac{2}{10}=0.2.
$$

The density is concentrated near $0$, so this distribution represents a prior belief that the advertisement is likely to have a low CTR. It is right-skewed because the long tail extends towards larger CTR values.

For $\text{Beta}(8,2)$,

$$
\mathbb{E}[\Theta]=\frac{8}{10}=0.8.
$$

The density is concentrated near $1$, representing a prior belief that the advertisement is likely to have a high CTR. It is left-skewed because the tail extends towards smaller CTR values.

Therefore, increasing $\alpha$ relative to $\beta$ shifts the density towards $1$, while increasing $\beta$ relative to $\alpha$ shifts the density towards $0$.

## Task 2: Sequential Likelihood and Joint History

Given the unknown CTR $\theta$, a single user interaction $Y_k$ follows a Bernoulli distribution. Therefore, the likelihood contribution of one observation $y_k\in\{0,1\}$ is

$$
L(y_k\mid\theta)
=
P(Y_k=y_k\mid\Theta=\theta)
=
\theta^{y_k}(1-\theta)^{1-y_k}.
$$

This expression covers both possible outcomes:

$$
L(y_k\mid\theta)
=
\begin{cases}
\theta, & \text{if } y_k=1,\\
1-\theta, & \text{if } y_k=0.
\end{cases}
$$

Assuming that interactions are conditionally independent given $\Theta=\theta$, the joint likelihood for the running interaction history

$$
\mathbf{y}^{(k)}=(y_1,y_2,\ldots,y_k)
$$

is

$$
L(\mathbf{y}^{(k)}\mid\theta)
=
\prod_{j=1}^{k}
\theta^{y_j}(1-\theta)^{1-y_j}.
$$

This can also be written as

$$
L(\mathbf{y}^{(k)}\mid\theta)
=
\theta^{\sum_{j=1}^{k}y_j}
(1-\theta)^{k-\sum_{j=1}^{k}y_j}.
$$

Here, $\sum_{j=1}^{k}y_j$ is the total number of clicks and $k-\sum_{j=1}^{k}y_j$ is the total number of non-clicks observed up to step $k$.

## Task 3: Closed-Form Analytical Updates (Conjugacy)

Suppose that, after step $k-1$, the current posterior is

$$
\Theta\mid\mathbf{Y}^{(k-1)}=\mathbf{y}^{(k-1)}
\sim
\text{Beta}(\alpha_{k-1},\beta_{k-1}).
$$

Its density is proportional to

$$
f_{\Theta\mid\mathbf{Y}^{(k-1)}}
(\theta\mid\mathbf{y}^{(k-1)})
\propto
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1}.
$$

After observing a new interaction $y_k$, Bayes' theorem gives

$$
f_{\Theta\mid\mathbf{Y}^{(k)}}
(\theta\mid\mathbf{y}^{(k)})
\propto
L(y_k\mid\theta)
f_{\Theta\mid\mathbf{Y}^{(k-1)}}
(\theta\mid\mathbf{y}^{(k-1)}).
$$

Substituting the Bernoulli likelihood gives

$$
f_{\Theta\mid\mathbf{Y}^{(k)}}
(\theta\mid\mathbf{y}^{(k)})
\propto
\theta^{y_k}(1-\theta)^{1-y_k}
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1}.
$$

Combining powers produces

$$
f_{\Theta\mid\mathbf{Y}^{(k)}}
(\theta\mid\mathbf{y}^{(k)})
\propto
\theta^{\alpha_{k-1}+y_k-1}
(1-\theta)^{\beta_{k-1}+1-y_k-1}.
$$

This has the form of a Beta distribution. Therefore,

$$
\Theta\mid\mathbf{Y}^{(k)}=\mathbf{y}^{(k)}
\sim
\text{Beta}(\alpha_k,\beta_k),
$$

where the recursive updates are

$$
\alpha_k=\alpha_{k-1}+y_k,
$$

and

$$
\beta_k=\beta_{k-1}+1-y_k.
$$

Thus, a click increases $\alpha$ by one, while a non-click increases $\beta$ by one. This is called Beta-Binomial conjugacy because the Beta prior and Bernoulli/Binomial likelihood produce a Beta posterior.

The posterior mean at step $k$ is

$$
\widehat{\theta}_{\mathrm{Bayes}}^{(k)}
=
\mathbb{E}[\Theta\mid\mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]
=
\frac{\alpha_k}{\alpha_k+\beta_k}.
$$

## Task 4: Dynamic Shifting Mechanics

If a click is observed, then $y_k=1$. The parameter updates are

$$
\alpha_k=\alpha_{k-1}+1,
\qquad
\beta_k=\beta_{k-1}.
$$

The posterior density is shifted towards larger values of $\theta$, because a click is evidence that the true CTR is higher.

If a non-click is observed, then $y_k=0$. The parameter updates are

$$
\alpha_k=\alpha_{k-1},
\qquad
\beta_k=\beta_{k-1}+1.
$$

The posterior density is shifted towards smaller values of $\theta$, because a non-click is evidence that the true CTR is lower.

This Beta-Bernoulli model has an analytical closed-form update because the Beta prior is conjugate to the Bernoulli likelihood. Only the two shape parameters $\alpha$ and $\beta$ need to be stored and updated.

In contrast, the 2PL item response model uses a logistic likelihood together with a normal prior. This pair is non-conjugate, so its posterior does not have a simple closed form. Consequently, numerical methods such as a fixed grid approximation and numerical integration are required for the 2PL model.

## Task 5: Running Point Estimators

At step $k$, the posterior distribution is

$$
\Theta\mid\mathbf{Y}^{(k)}=\mathbf{y}^{(k)}
\sim
\text{Beta}(\alpha_k,\beta_k).
$$

The running posterior mean is

$$
\widehat{\theta}_{\mathrm{Bayes}}^{(k)}
=
\frac{\alpha_k}{\alpha_k+\beta_k}.
$$

When $\alpha_k>1$ and $\beta_k>1$, the MAP estimate is the interior mode of the Beta distribution:

$$
\widehat{\theta}_{\mathrm{MAP}}^{(k)}
=
\frac{\alpha_k-1}{\alpha_k+\beta_k-2}.
$$

For boundary cases:

$$
\widehat{\theta}_{\mathrm{MAP}}^{(k)}
=
0
\quad
\text{if }
\alpha_k\leq1
\text{ and }
\beta_k>1,
$$

and

$$
\widehat{\theta}_{\mathrm{MAP}}^{(k)}
=
1
\quad
\text{if }
\alpha_k>1
\text{ and }
\beta_k\leq1.
$$

For the initial uniform prior $\text{Beta}(1,1)$, every value in $[0,1]$ has equal density. Therefore, the MAP estimate at step $0$ is not unique.

## Task 6: Performance Tracking and Convergence Analysis

The true CTR is assumed to be

$$
\theta_{\text{true}}=0.35.
$$

The initial prior is uniform:

$$
\Theta\sim\text{Beta}(1,1),
$$

so that

$$
\alpha_0=1,
\qquad
\beta_0=1.
$$

A total of $n=100$ impressions are simulated. At each step, the user clicks with probability $\theta_{\text{true}}$, and the Beta posterior parameters are updated analytically.

In [21]:
import numpy as np
import plotly.graph_objects as go

# Reproducible simulation
rng = np.random.default_rng(seed=42)

# True CTR and number of impressions
theta_true = 0.35
n_impressions = 100

# Initial uniform prior: Beta(1, 1)
alpha = 1
beta_param = 1

# Store estimates from step 0 onwards
steps = [0]
posterior_means = [alpha / (alpha + beta_param)]

# The uniform Beta(1, 1) prior has no unique MAP estimate.
# NaN prevents an arbitrary value from being plotted at step 0.
map_estimates = [np.nan]

responses = []

def beta_map(alpha, beta_param):
    """Return the MAP estimate of a Beta(alpha, beta) distribution."""
    if alpha > 1 and beta_param > 1:
        return (alpha - 1) / (alpha + beta_param - 2)
    elif alpha <= 1 and beta_param > 1:
        return 0.0
    elif alpha > 1 and beta_param <= 1:
        return 1.0
    else:
        return np.nan  # Uniform case: no unique mode

# Sequentially simulate impressions and update Beta parameters
for k in range(1, n_impressions + 1):
    # Simulate one click/non-click response
    y_k = int(rng.uniform(0, 1) < theta_true)
    responses.append(y_k)

    # Conjugate Beta-Bernoulli update
    alpha = alpha + y_k
    beta_param = beta_param + (1 - y_k)

    # Store the posterior mean and MAP
    posterior_means.append(alpha / (alpha + beta_param))
    map_estimates.append(beta_map(alpha, beta_param))
    steps.append(k)

print("Total clicks:", sum(responses))
print("Observed CTR:", sum(responses) / n_impressions)
print("Final posterior: Beta({}, {})".format(alpha, beta_param))

# Create convergence plot
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode="lines",
        name="Posterior Mean",
        line=dict(width=3)
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines",
        name="MAP Estimate",
        line=dict(width=3, dash="dash")
    )
)

fig.add_hline(
    y=theta_true,
    line_dash="dot",
    line_color="red",
    annotation_text="True CTR = 0.35"
)

fig.update_layout(
    title="Sequential CTR Estimation Using Beta-Binomial Updates",
    xaxis_title="Number of impressions, k",
    yaxis_title="Estimated click-through rate",
    yaxis=dict(range=[0, 1]),
    template="plotly_white",
    width=900,
    height=550,
    legend_title="Estimator"
)

fig.show()

Total clicks: 33
Observed CTR: 0.33
Final posterior: Beta(34, 68)


### Analysis

At the beginning, the prior is $\text{Beta}(1,1)$, which is uniform and has posterior mean

$$
\widehat{\theta}_{\mathrm{Bayes}}^{(0)}=0.5.
$$

This initial estimate differs from the true CTR,

$$
\theta_{\text{true}}=0.35.
$$

As more impressions are observed, the influence of the initial prior becomes weaker relative to the accumulated click and non-click evidence. Consequently, both the posterior mean and MAP estimate generally move towards the true CTR.

The estimates may fluctuate during the first few impressions because the sample contains random variation. As the number of impressions approaches $100$, the estimates should become more stable and usually remain closer to $0.35$.

This behaviour demonstrates that Bayesian sequential updating accumulates evidence over time. The prior has the greatest influence when there are few observations, whereas the observed data increasingly dominate the posterior as $k$ becomes large.

# Bayesian Estimation for Structural Health Monitoring via Bounded Grid Updates

Let $\Theta=\theta$ represent the remaining stiffness efficiency factor of a structural component, where

$$
\theta\in(0,1].
$$

A value of $\theta=1$ represents a fully healthy component, while a value close to $0$ represents severe structural degradation.

At inspection step $k$, the measured stiffness is modelled as

$$
y_k
=
\theta K_{\text{nominal}}e^{\epsilon_k},
\qquad
\epsilon_k\sim\mathscr{N}(0,\sigma^2).
$$

Thus, conditional on $\Theta=\theta$, the measurement $Y_k$ follows a log-normal distribution.

## Task 1: Prior Belief Boundaries

Before collecting sensor measurements, the structural stiffness factor is assigned the prior distribution

$$
\Theta\sim\text{Beta}(8,1.5).
$$

This prior is defined on the physically meaningful interval $[0,1]$.

In [22]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Restricted physical grid: theta cannot be zero
theta_grid = np.linspace(0.01, 1.0, 1000)

# Beta(8, 1.5) prior density
prior_density = beta.pdf(theta_grid, a=8, b=1.5)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=theta_grid,
        y=prior_density,
        mode="lines",
        name="Beta(8, 1.5) prior",
        line=dict(width=3, color="royalblue"),
        fill="tozeroy"
    )
)

fig.update_layout(
    title="Prior Distribution of Remaining Stiffness Efficiency",
    xaxis_title="Stiffness efficiency factor, θ",
    yaxis_title="Prior density",
    template="plotly_white",
    width=900,
    height=550
)

fig.show()

### Interpretation

For a Beta distribution,

$$
\mathbb{E}[\Theta]
=
\frac{\alpha}{\alpha+\beta}.
$$

Therefore, for the prior $\Theta\sim\text{Beta}(8,1.5)$,

$$
\mathbb{E}[\Theta^{(0)}]
=
\frac{8}{8+1.5}
=
\frac{8}{9.5}
\approx 0.842.
$$

The prior mean is approximately $0.842$, meaning that the component is initially expected to retain about $84.2\%$ of its nominal stiffness.

This is an appropriate prior for a component believed to be healthy because most of the prior density is concentrated near $\theta=1$. However, the distribution still allows smaller values of $\theta$, so it can be revised if sensor readings provide evidence of structural damage.

## Task 2: Structural Likelihood Formulation

The measurement model is

$$
Y_k
=
\theta K_{\text{nominal}}e^{\epsilon_k},
\qquad
\epsilon_k\sim\mathscr{N}(0,\sigma^2).
$$

Taking logarithms gives

$$
\log Y_k
=
\log(\theta K_{\text{nominal}})
+
\epsilon_k.
$$

Therefore,

$$
\log Y_k\mid\Theta=\theta
\sim
\mathscr{N}
\left(
\log(\theta K_{\text{nominal}}),
\sigma^2
\right).
$$

Hence, the likelihood contribution of a single positive sensor measurement $y_k$ is the log-normal density:

$$
L(y_k\mid\theta)
=
\frac{1}{y_k\sigma\sqrt{2\pi}}
\exp\left[
-\frac{
\left(
\log y_k-\log(\theta K_{\text{nominal}})
\right)^2
}{
2\sigma^2
}
\right].
$$

Assuming that sensor readings are conditionally independent given $\Theta=\theta$, the joint likelihood for the running observation history

$$
\mathbf{y}^{(k)}
=
(y_1,y_2,\ldots,y_k)
$$

is

$$
L(\mathbf{y}^{(k)}\mid\theta)
=
\prod_{j=1}^{k}
\frac{1}{y_j\sigma\sqrt{2\pi}}
\exp\left[
-\frac{
\left(
\log y_j-\log(\theta K_{\text{nominal}})
\right)^2
}{
2\sigma^2
}
\right].
$$

## Task 3: Mathematical Formulation of the Non-Conjugate Grid Update

The prior distribution is Beta:

$$
f_{\Theta}^{(0)}(\theta)
\propto
\theta^{\alpha_0-1}(1-\theta)^{\beta_0-1}.
$$

However, the log-normal likelihood contains the term

$$
\exp\left[
-\frac{
\left(
\log y_k-\log(\theta K_{\text{nominal}})
\right)^2
}{
2\sigma^2
}
\right].
$$

Multiplying this likelihood by a Beta density does not produce another Beta density. Therefore, the Beta prior and log-normal likelihood are not conjugate, and no simple closed-form posterior distribution is available.

The posterior at step $k$ is updated recursively as

$$
f_{\Theta\mid\mathbf{Y}^{(k)}}
(\theta\mid\mathbf{y}^{(k)})
\propto
L(y_k\mid\theta)
f_{\Theta\mid\mathbf{Y}^{(k-1)}}
(\theta\mid\mathbf{y}^{(k-1)}).
$$

Substituting the log-normal likelihood gives

$$
f_{\Theta\mid\mathbf{Y}^{(k)}}
(\theta\mid\mathbf{y}^{(k)})
\propto
\frac{1}{y_k\sigma\sqrt{2\pi}}
\exp\left[
-\frac{
\left(
\log y_k-\log(\theta K_{\text{nominal}})
\right)^2
}{
2\sigma^2
}
\right]
f_{\Theta\mid\mathbf{Y}^{(k-1)}}
(\theta\mid\mathbf{y}^{(k-1)}).
$$

The posterior must therefore be approximated numerically on a grid of values in the bounded domain

$$
\theta\in(0,1].
$$

## Task 4: Running Point Estimates

Let

$$
f_{\Theta\mid\mathbf{Y}^{(k)}}
(\theta\mid\mathbf{y}^{(k)})
$$

be the normalized posterior density after $k$ sensor readings.

The running posterior mean is

$$
\widehat{\theta}_{\mathrm{Bayes}}^{(k)}
=
\mathbb{E}
[
\Theta\mid\mathbf{Y}^{(k)}=\mathbf{y}^{(k)}
]
=
\int_0^1
\theta
f_{\Theta\mid\mathbf{Y}^{(k)}}
(\theta\mid\mathbf{y}^{(k)})
\,d\theta.
$$

The running MAP estimate is the value of $\theta$ that maximizes the posterior density:

$$
\widehat{\theta}_{\mathrm{MAP}}^{(k)}
=
\underset{\theta\in(0,1]}{\operatorname{arg\,max}}
\;
f_{\Theta\mid\mathbf{Y}^{(k)}}
(\theta\mid\mathbf{y}^{(k)}).
$$

Because a closed-form posterior is unavailable, these quantities are evaluated numerically. The posterior mean is approximated using numerical integration, and the MAP estimate is approximated by selecting the grid point with the largest posterior density.

## Task 5: Algorithmic Grid Approximation and Normalization

A numerical grid algorithm can be used to sequentially approximate the posterior distribution.

1. Define a grid of possible stiffness factors:

$$
\theta_1,\theta_2,\ldots,\theta_m
\in[0.01,1.0].
$$

The value $0$ is avoided because $\log(\theta)$ is undefined at $\theta=0$.

2. Evaluate the initial Beta prior at each grid point:

$$
f^{(0)}(\theta_g)
=
\text{BetaPDF}(\theta_g;8,1.5).
$$

3. Normalize the prior numerically using the trapezoidal rule:

$$
f^{(0)}(\theta_g)
\leftarrow
\frac{
f^{(0)}(\theta_g)
}{
\texttt{np.trapezoid}(f^{(0)},\theta_{\text{grid}})
}.
$$

4. When a new measurement $y_k$ arrives, calculate the log-normal likelihood at every grid point:

$$
L(y_k\mid\theta_g)
=
\frac{1}{y_k\sigma\sqrt{2\pi}}
\exp\left[
-\frac{
\left(
\log y_k-\log(\theta_g K_{\text{nominal}})
\right)^2
}{
2\sigma^2
}
\right].
$$

5. Form the unnormalized posterior:

$$
\widetilde f^{(k)}(\theta_g)
=
L(y_k\mid\theta_g)
f^{(k-1)}(\theta_g).
$$

6. Compute the numerical normalizing constant:

$$
Z_k
\approx
\texttt{np.trapezoid}
(
\widetilde f^{(k)},
\theta_{\text{grid}}
).
$$

7. Normalize the updated density:

$$
f^{(k)}(\theta_g)
=
\frac{
\widetilde f^{(k)}(\theta_g)
}{
Z_k
}.
$$

The normalized posterior is then used as the prior for the next sensor reading.

## Task 6: Performance Tracking and Degradation Convergence Analysis

Suppose that an impact reduces the true remaining stiffness efficiency to

$$
\theta_{\text{true}}=0.68.
$$

The monitoring system uses

$$
K_{\text{nominal}}=50.0\text{ kN/mm},
\qquad
\sigma=0.15,
\qquad
n=15.
$$

At each step, a sensor measurement is simulated from

$$
Y_k
=
\theta_{\text{true}}K_{\text{nominal}}e^{\epsilon_k},
\qquad
\epsilon_k\sim\mathscr{N}(0,\sigma^2).
$$

The simulation tracks the full posterior distribution, the posterior mean, and the MAP estimate after every new reading.

In [23]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Reproducible random simulation
rng = np.random.default_rng(seed=42)

# Model settings
theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
n_measurements = 15

# Bounded physical grid; theta = 0 is excluded because log(0) is undefined
theta_grid = np.linspace(0.01, 1.0, 2001)

# Initial prior: Beta(8, 1.5)
posterior = beta.pdf(theta_grid, a=8, b=1.5)
posterior = posterior / np.trapezoid(posterior, theta_grid)

def lognormal_likelihood(y, theta_grid, K_nominal, sigma):
    """Likelihood of one stiffness measurement for every theta grid value."""
    log_residual = np.log(y) - np.log(theta_grid * K_nominal)

    return (
        1 / (y * sigma * np.sqrt(2 * np.pi))
    ) * np.exp(
        -(log_residual ** 2) / (2 * sigma ** 2)
    )

# Values to retain for analysis and plotting
milestones = [0, 1, 2, 5, 10, 15]
posterior_curves = {0: posterior.copy()}

steps = [0]
posterior_means = [
    np.trapezoid(theta_grid * posterior, theta_grid)
]
map_estimates = [
    theta_grid[np.argmax(posterior)]
]
posterior_standard_deviations = [
    np.sqrt(
        np.trapezoid(
            ((theta_grid - posterior_means[-1]) ** 2) * posterior,
            theta_grid
        )
    )
]

sensor_readings = []

# Sequentially simulate readings and update the grid posterior
for k in range(1, n_measurements + 1):
    # Simulate a positive log-normal sensor measurement
    epsilon_k = rng.normal(loc=0, scale=sigma)
    y_k = theta_true * K_nominal * np.exp(epsilon_k)
    sensor_readings.append(y_k)

    # Calculate likelihood and unnormalized posterior
    likelihood = lognormal_likelihood(
        y_k,
        theta_grid,
        K_nominal,
        sigma
    )
    unnormalized_posterior = likelihood * posterior

    # Normalize using the trapezoidal rule
    normalizing_constant = np.trapezoid(
        unnormalized_posterior,
        theta_grid
    )
    posterior = unnormalized_posterior / normalizing_constant

    # Calculate running estimators
    posterior_mean = np.trapezoid(
        theta_grid * posterior,
        theta_grid
    )
    map_estimate = theta_grid[np.argmax(posterior)]

    posterior_variance = np.trapezoid(
        ((theta_grid - posterior_mean) ** 2) * posterior,
        theta_grid
    )

    # Store results
    steps.append(k)
    posterior_means.append(posterior_mean)
    map_estimates.append(map_estimate)
    posterior_standard_deviations.append(np.sqrt(posterior_variance))

    if k in milestones:
        posterior_curves[k] = posterior.copy()

# Display simulated readings and final estimates
print("Simulated sensor readings (kN/mm):")
print(np.round(sensor_readings, 3))

print("\nFinal posterior mean:", round(posterior_means[-1], 4))
print("Final MAP estimate:", round(map_estimates[-1], 4))
print("Final posterior standard deviation:",
      round(posterior_standard_deviations[-1], 4))

# Plot 1: posterior density curves at selected milestones
fig_posterior = go.Figure()

for k in milestones:
    fig_posterior.add_trace(
        go.Scatter(
            x=theta_grid,
            y=posterior_curves[k],
            mode="lines",
            name=f"Step {k}",
            line=dict(width=2)
        )
    )

fig_posterior.add_vline(
    x=theta_true,
    line_dash="dot",
    line_color="red",
    annotation_text="True stiffness = 0.68"
)

fig_posterior.update_layout(
    title="Sequential Posterior Density Curves",
    xaxis_title="Remaining stiffness efficiency, θ",
    yaxis_title="Posterior density",
    template="plotly_white",
    width=900,
    height=550,
    legend_title="Posterior milestone"
)

fig_posterior.show()

# Plot 2: convergence of posterior mean and MAP estimate
fig_estimators = go.Figure()

fig_estimators.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode="lines+markers",
        name="Posterior Mean",
        line=dict(width=3)
    )
)

fig_estimators.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP Estimate",
        line=dict(width=3, dash="dash")
    )
)

fig_estimators.add_hline(
    y=theta_true,
    line_dash="dot",
    line_color="red",
    annotation_text="True stiffness = 0.68"
)

fig_estimators.update_layout(
    title="Convergence of Sequential Stiffness Estimates",
    xaxis_title="Number of sensor readings, k",
    yaxis_title="Estimated remaining stiffness efficiency",
    template="plotly_white",
    width=900,
    height=550,
    legend_title="Estimator"
)

fig_estimators.show()

Simulated sensor readings (kN/mm):
[35.59  29.089 38.051 39.152 25.373 27.967 34.658 32.425 33.914 29.916
 38.794 38.207 34.338 40.264 36.47 ]

Final posterior mean: 0.6873
Final MAP estimate: 0.6857
Final posterior standard deviation: 0.0266


### Analysis

Initially, the prior $\text{Beta}(8,1.5)$ is concentrated near high values of $\theta$, reflecting an optimistic belief that the component is healthy. Its prior mean is approximately $0.842$, which is above the true damaged state:

$$
\theta_{\text{true}}=0.68.
$$

As sensor readings are received, measurements that are lower than the healthy nominal stiffness provide evidence for a smaller value of $\theta$. Therefore, the posterior mean and MAP estimate shift downwards from the initial optimistic prior towards the true stiffness efficiency.

The exact number of readings required to isolate the damaged state depends on the random sensor noise in the simulated timeline. However, the plots should show that the posterior moves substantially towards $0.68$ after the first few readings and becomes progressively narrower as more readings are incorporated.

The narrowing posterior curves indicate decreasing uncertainty about the stiffness factor. From a structural-safety perspective, this is important because the monitoring system becomes increasingly confident in whether the component lies below a chosen safety threshold. A posterior concentrated around $0.68$ provides strong evidence that the component has experienced meaningful degradation relative to its healthy state.